# OCEAN GUARD AI · entrenamiento final en Colab (GPU)

Mismo reparto 70/15/15 (por escenas / bloques / regiones) y mismo preprocesado que el entrenamiento local; parte de
`unet_resnet34_demo_12ep/best.pt`. **Antes de empezar:** suba `paquete_colab.zip` a `Mi unidad/OCEANGUARD/` en Google Drive y
elija *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Ejecute las celdas en orden.
Resultados en `Mi unidad/OCEANGUARD/resultado/`: `best.pt`, `bn_decoder_recal.pt`, `historial.json`, `metricas_test.json`.

In [ ]:
# 1) Montar Drive y descomprimir el paquete
from google.colab import drive
drive.mount('/content/drive')
import zipfile, pathlib
ZIP = '/content/drive/MyDrive/OCEANGUARD/paquete_colab.zip'
DATA = pathlib.Path('/content/ogdata')
RESULT = pathlib.Path('/content/drive/MyDrive/OCEANGUARD/resultado')
RESULT.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA)
print(sorted(p.name for p in DATA.iterdir()))

In [ ]:
# 2) Verificar GPU e instalar la librería del modelo
%pip -q install segmentation-models-pytorch==0.6.0
import torch
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU'
DEVICE = 'cuda'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3) Parámetros (los mismos del entrenamiento local; cambian solo las épocas y la parada temprana)
import sys, json, time, copy, random
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import segmentation_models_pytorch as smp

EPOCHS_MAX, PATIENCE = 15, 3            # máximo de épocas y parada temprana por validación
BATCH, SEED = 4, 42
LR_ENCODER, LR_DECODER, WEIGHT_DECAY = 1e-5, 1e-4, 1e-4
THRESHOLD, MIN_PX = 0.95, 100            # regla de la plataforma: prob > 0,95 y zonas >= 100 px
NUM_WORKERS = 2
MAX_TRAIN_BATCHES = None                 # solo para pruebas rápidas (None = época completa)
EVAL_LIMIT = None                        # solo para pruebas rápidas (None = todas las imágenes)

sys.path.insert(0, str(DATA / 'codigo'))
import data as D                          # preprocesado del proyecto: normalización ImageNet, recorte 256, padding a múltiplo de 32
from train import dice_bce_loss, set_seed
import analysis as A                      # remove_small_components (igual que la plataforma)

SPLIT = json.load(open(DATA / 'split.json', encoding='utf-8'))
print({n: SPLIT['counts'][n] for n in ('train', 'val', 'test')})

In [ ]:
# 4) Datos: pares imagen/máscara (gris en 3 canales, ImageNet, máscara binaria)
def pairs(split):
    out = []
    for p in sorted((DATA / split / 'images').iterdir()):
        out.append((p, DATA / split / 'masks' / (p.stem + '.png'), p.stem.split('_')[0]))   # fuente: orig | sos | csiro
    return out

def read_gray_rgb(path):
    a = np.asarray(Image.open(path).convert('RGB'))
    return np.stack([a[..., 0]] * 3, axis=-1)                                 # gris repetido en 3 canales

def read_mask(path):
    return np.asarray(Image.open(path).convert('L')) > 127

def augment(img, msk, rng):
    k = int(rng.integers(0, 4))
    img, msk = np.rot90(img, k), np.rot90(msk, k)
    if rng.integers(0, 2):
        img, msk = img[:, ::-1], msk[:, ::-1]
    return np.ascontiguousarray(img), np.ascontiguousarray(msk)

class TrainSet(Dataset):
    def __init__(self, items, seed):
        self.items, self.seed, self.epoch = items, seed, 0
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        ip, mp, _ = self.items[idx]
        rng = np.random.default_rng([self.seed, self.epoch, idx])
        img = (read_gray_rgb(ip).astype(np.float32) / 255.0 - D.IMAGENET_MEAN) / D.IMAGENET_STD
        mask = read_mask(mp).astype(np.float32)
        if mask.shape[0] > D.CROP_SIZE or mask.shape[1] > D.CROP_SIZE:
            top, left = D.random_crop_coords(mask.shape[0], mask.shape[1], D.CROP_SIZE, rng)
            img, mask = D.crop_pair(img, mask, top, left, D.CROP_SIZE)
        img, mask = augment(img, mask, rng)
        return {'image': torch.from_numpy(img).permute(2, 0, 1).contiguous(), 'mask': torch.from_numpy(mask).unsqueeze(0).contiguous()}

train_items, val_items, test_items = pairs('train'), pairs('val'), pairs('test')
print('imágenes -> train:', len(train_items), '| val:', len(val_items), '| test:', len(test_items))

In [ ]:
# 5) Modelo (punto de partida) y evaluación con la regla de la plataforma
def build_model():
    m = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=3, classes=1)
    return m

TILE, OVERLAP = 512, 64
@torch.no_grad()
def predict_prob(model, rgb):
    h, w = rgb.shape[:2]
    arr = (rgb.astype(np.float32) / 255.0 - D.IMAGENET_MEAN) / D.IMAGENET_STD
    pads = D.pad_amounts(h, w, D.MULTIPLE)
    padded = D.pad_hw(arr, pads)
    def fwd(a):
        x = torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        return torch.sigmoid(model(x))[0, 0].cpu().numpy()
    if max(h, w) <= TILE:
        prob = fwd(padded)
    else:                                    # imágenes grandes: mosaicos con solape, como la plataforma
        H, W = padded.shape[:2]
        acc, cnt = np.zeros((H, W), np.float32), np.zeros((H, W), np.float32)
        step = TILE - OVERLAP
        for y in sorted(set(list(range(0, H - TILE + 1, step)) + [H - TILE])):
            for x in sorted(set(list(range(0, W - TILE + 1, step)) + [W - TILE])):
                acc[y:y + TILE, x:x + TILE] += fwd(padded[y:y + TILE, x:x + TILE]); cnt[y:y + TILE, x:x + TILE] += 1
        prob = acc / np.maximum(cnt, 1)
    return D.crop_hw(prob, pads).astype(np.float32)

def _acc():
    return {'tp': 0, 'fp': 0, 'fn': 0, 'img': 0, 'marked': 0}

def _fin(a):
    return {'imagenes': a['img'], 'dice': 2 * a['tp'] / max(2 * a['tp'] + a['fp'] + a['fn'], 1),
            'precision': a['tp'] / max(a['tp'] + a['fp'], 1), 'recall': a['tp'] / max(a['tp'] + a['fn'], 1),
            'imagenes_marcadas': a['marked'], 'tasa_marcadas': a['marked'] / a['img'] if a['img'] else None}

def evaluate(model, items):
    model.eval()
    items = items[::max(1, len(items) // EVAL_LIMIT)] if EVAL_LIMIT else items   # EVAL_LIMIT: solo para pruebas rápidas
    spill, clean, full, src = _acc(), _acc(), _acc(), {}
    for ip, mp, s in items:
        mask = read_mask(mp)
        m = A.remove_small_components(predict_prob(model, read_gray_rgb(ip)) > THRESHOLD, MIN_PX)
        marked = bool(m.any())
        tp, fp, fn = int((m & mask).sum()), int((m & ~mask).sum()), int((~m & mask).sum())
        for acc in (spill if mask.any() else clean, full):
            acc['tp'] += tp; acc['fp'] += fp; acc['fn'] += fn; acc['img'] += 1; acc['marked'] += int(marked)
        if mask.any():
            a = src.setdefault(s, _acc()); a['tp'] += tp; a['fp'] += fp; a['fn'] += fn; a['img'] += 1; a['marked'] += int(marked)
    out = {'con_derrame': _fin(spill), 'sin_derrame': _fin(clean), 'conjunto_completo': _fin(full),
           'con_derrame_por_fuente': {k: _fin(v) for k, v in src.items()}}
    out['con_derrame']['deteccion'] = out['con_derrame'].pop('tasa_marcadas')
    out['sin_derrame']['falsa_alarma'] = out['sin_derrame'].pop('tasa_marcadas')
    for k in ('dice', 'precision', 'recall'):
        out['sin_derrame'].pop(k)
    return out

def recalibrate_bn(model, items):
    # igual que platform/tools/recalibrate_bn.py: estadísticas BatchNorm del decoder con imágenes de ENTRENAMIENTO, sin etiquetas
    bns = [m for n, m in model.named_modules() if isinstance(m, torch.nn.BatchNorm2d) and not n.startswith('encoder.')]
    for b in bns:
        b.reset_running_stats(); b.momentum = None
    model.eval()
    for b in bns:
        b.train()
    with torch.no_grad():
        for ip, _, _ in items:
            arr = (read_gray_rgb(ip).astype(np.float32) / 255.0 - D.IMAGENET_MEAN) / D.IMAGENET_STD
            pads = D.pad_amounts(*arr.shape[:2], D.MULTIPLE)
            model(torch.from_numpy(np.ascontiguousarray(D.pad_hw(arr, pads))).permute(2, 0, 1).unsqueeze(0).to(DEVICE))
    model.eval()
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            if not k.startswith('encoder.') and ('running_mean' in k or 'running_var' in k or 'num_batches_tracked' in k)}

In [ ]:
# 6) Entrenar (hasta 15 épocas, parada temprana con paciencia 3 por validación)
set_seed(SEED)
model = build_model()
ck = torch.load(DATA / 'unet_resnet34_demo_12ep' / 'best.pt', map_location='cpu', weights_only=False)
model.load_state_dict(ck['model'], strict=True)          # punto de partida: unet_resnet34_demo_12ep/best.pt
model.to(DEVICE)

enc_ids = {id(p) for p in model.encoder.parameters()}
enc_params = list(model.encoder.parameters())
dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
opt = torch.optim.AdamW([{'params': enc_params, 'lr': LR_ENCODER}, {'params': dec_params, 'lr': LR_DECODER}], weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_MAX)
ds = TrainSet(train_items, SEED)

def score_of(v):                                          # criterio de selección (mismo que el entrenamiento local)
    return v['conjunto_completo']['dice'] - 0.5 * v['sin_derrame']['falsa_alarma']

history, best, bad = [], None, 0
print(f'train {len(train_items)} | val {len(val_items)} | lote {BATCH} | máx. {EPOCHS_MAX} épocas | paciencia {PATIENCE}', flush=True)
for ep in range(1, EPOCHS_MAX + 1):
    ds.epoch = ep
    t0 = time.time()
    model.train()
    tot = n = 0
    loader = DataLoader(ds, batch_size=BATCH, shuffle=True, generator=torch.Generator().manual_seed(SEED + ep),
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))
    for b in loader:
        opt.zero_grad(set_to_none=True)
        loss = dice_bce_loss(model(b['image'].to(DEVICE)), b['mask'].to(DEVICE))
        loss.backward()
        opt.step()
        tot += loss.item(); n += 1
        if MAX_TRAIN_BATCHES and n >= MAX_TRAIN_BATCHES:
            break
    sched.step()
    val = evaluate(model, val_items)
    score = score_of(val)
    rec = {'epoch': ep, 'train_loss': tot / n, 'score_seleccion': score, 'val': val, 'segundos': round(time.time() - t0)}
    history.append(rec)
    mejora = best is None or score > best['score']
    if mejora:
        best, bad = {'epoch': ep, 'score': score}, 0
        torch.save({'global_epoch': ep, 'phase': 'final_colab', 'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
                    'score': score}, RESULT / 'best.pt')
    else:
        bad += 1
    json.dump({'historial': history, 'mejor': best, 'parametros': {'epocas_max': EPOCHS_MAX, 'paciencia': PATIENCE, 'lote': BATCH,
               'lr_encoder': LR_ENCODER, 'lr_decoder': LR_DECODER, 'semilla': SEED}}, open(RESULT / 'historial.json', 'w'), indent=1)
    print(f"época {ep:2d}/{EPOCHS_MAX} | pérdida {tot/n:.4f} | val: Dice {val['conjunto_completo']['dice']:.4f}, "
          f"falsa alarma {val['sin_derrame']['falsa_alarma']:.3f}, detección {val['con_derrame']['deteccion']:.3f} | "
          f"puntaje {score:.4f}{'  ← mejor' if mejora else f'  (sin mejora {bad}/{PATIENCE})'} | {rec['segundos']} s", flush=True)
    if bad >= PATIENCE:
        print(f'Parada temprana en la época {ep}: sin mejora en {PATIENCE} épocas. Mejor: época {best["epoch"]}.')
        break

In [ ]:
# 7) Recalibrar BatchNorm del mejor modelo y evaluar UNA sola vez en test
ck = torch.load(RESULT / 'best.pt', map_location='cpu', weights_only=False)
model.load_state_dict(ck['model'])
model.to(DEVICE)
bn = recalibrate_bn(model, [t for t in train_items if t[2] == 'orig'])      # solo originales de entrenamiento
torch.save({'state': bn, 'n_images': sum(1 for t in train_items if t[2] == 'orig'), 'split': 'train(final)'}, RESULT / 'bn_decoder_recal.pt')
test = evaluate(model, test_items)                                         # prueba: una sola vez
res = {'regla': {'umbral': THRESHOLD, 'zona_minima_px': MIN_PX}, 'mejor_epoca': best['epoch'],
       'imagenes_prueba': len(test_items), 'metricas_test': test}
json.dump(res, open(RESULT / 'metricas_test.json', 'w'), indent=1)
print(json.dumps({k: {m: (round(x, 4) if isinstance(x, float) else x) for m, x in v.items() if not isinstance(x, dict)}
                  for k, v in test.items() if k != 'con_derrame_por_fuente'}, indent=1))
print('Guardado en', RESULT)